# Did AI Overviews reduce Wikipedia traffic?

## Background

Google AI Overviews have changed how users interact with search results. Instead of clicking through to external websites, users may receive an answer directly on the search results page.

Wikipedia is one of the world's largest open knowledge platforms. Its articles are widely used as references in search engines, AI systems, and other information services. At the same time, Wikimedia provides open access to pageview statistics, making it possible to study user behaviour using publicly available data.

This combination makes Wikipedia an excellent case for investigating whether AI-generated answers changed how people access information.

## **Research question**

**Did the launch of Google AI Overviews reduce traffic to selected Wikipedia articles?**

## Why this dataset?

I intentionally chose Wikipedia because the data are publicly available, reproducible, and globally relevant. Anyone can rerun this analysis using the same open datasets, making the research transparent and easy to verify.


## Treatment group selection

AI Overviews are not shown for every search query. Therefore, the treatment group should consist of topics where Google frequently displays AI-generated answers and where users often seek factual information.

Instead of selecting articles randomly, I manually verified whether Google AI Overviews appeared for each candidate query.

The treatment group consists of medical topics that satisfy all of the following criteria:

- Google displays an AI Overview.
- The query is informational rather than navigational.
- A corresponding English Wikipedia article exists.
- The topic is commonly searched and suitable for public traffic analysis.

The treatment group definition is stored in a separate Excel file to make the analysis reproducible and easy to modify.

In [19]:
treatment = pd.read_excel(
    "treatment_group_selection.xlsx",
    sheet_name="treatment_group"
)

print(f"Treatment articles: {len(treatment)}")

treatment.head()

Treatment articles: 15


,Query,Wikipedia article,Category,AI Overview,Primary source cited,Included,Selection rationale
0,Migraine,Migraine,Disease,Yes,Wikipedia,✅,Frequently searched medical condition with AI ...
1,Diabetes,Diabetes,Disease,Yes,WHO,✅,High-volume chronic disease query
2,Depression,Depression (mood),Mental health,Yes,Wikipedia,✅,High-volume mental health query
3,Common cold,Common cold,Infection,Yes,Wikipedia,✅,Common acute infection query
4,Influenza,Influenza,Infection,Yes,CDC,✅,Seasonal infectious disease


## Data collection

Wikipedia provides a public Pageviews API that allows anyone to retrieve daily traffic statistics for individual articles.

For each selected article, daily pageviews were collected from May 14, 2023 to November 14, 2024.

The intervention date was set to May 14, 2024, corresponding to Google's broader rollout of AI Overviews.

In [20]:
START_DATE = "2023-05-14"
END_DATE = "2024-11-14"

INTERVENTION_DATE = "2024-05-14"

In [21]:
ARTICLES = treatment["Wikipedia article"].tolist()

ARTICLES

['Migraine',
 'Diabetes',
 'Depression (mood)',
 'Common cold',
 'Influenza',
 'Asthma',
 'Hypertension',
 'Vitamin D deficiency',
 'Insomnia',
 'Anemia',
 'Fatigue',
 'Headache',
 'Iron deficiency',
 'Anxiety',
 'Pharyngitis']

In [29]:
import requests
import pandas as pd

def get_pageviews(article, start_date, end_date, project="en.wikipedia.org"):
    article_encoded = article.replace(" ", "_")
    start = start_date.replace("-", "")
    end = end_date.replace("-", "")

    url = (
        f"https://wikimedia.org/api/rest_v1/metrics/pageviews/per-article/"
        f"{project}/all-access/user/{article_encoded}/daily/{start}/{end}"
    )

    headers = {
        "User-Agent": "Wikipedia AI Overviews Research by Tamara Karavaya (tamkor19@gmail.com)"
    }

    response = requests.get(url, headers=headers)

    if response.status_code != 200:
        print(f"Failed: {article} | status code: {response.status_code}")
        return pd.DataFrame()

    data = response.json()["items"]

    df = pd.DataFrame(data)
    df["date"] = pd.to_datetime(df["timestamp"].str[:8])
    df["article"] = article
    df = df[["date", "article", "views"]]

    return df

In [30]:
test = get_pageviews("Migraine", START_DATE, END_DATE)
test.head()

,date,article,views
0,2023-05-14,Migraine,740
1,2023-05-15,Migraine,874
2,2023-05-16,Migraine,967
3,2023-05-17,Migraine,923
4,2023-05-18,Migraine,963


In [31]:
all_pageviews = []

for article in ARTICLES:
    print(f"Fetching: {article}")
    article_df = get_pageviews(article, START_DATE, END_DATE)
    all_pageviews.append(article_df)

raw_pageviews = pd.concat(all_pageviews, ignore_index=True)

print(f"Rows collected: {len(raw_pageviews)}")
raw_pageviews.head()

Fetching: Migraine
Fetching: Diabetes
Fetching: Depression (mood)
Fetching: Common cold
Fetching: Influenza
Fetching: Asthma
Fetching: Hypertension
Fetching: Vitamin D deficiency
Fetching: Insomnia
Fetching: Anemia
Fetching: Fatigue
Fetching: Headache
Fetching: Iron deficiency
Fetching: Anxiety
Fetching: Pharyngitis
Rows collected: 8265


,date,article,views
0,2023-05-14,Migraine,740
1,2023-05-15,Migraine,874
2,2023-05-16,Migraine,967
3,2023-05-17,Migraine,923
4,2023-05-18,Migraine,963


In [32]:
raw_pageviews.to_csv("raw_pageviews_data.csv", index=False)

print("Raw dataset saved.")

Raw dataset saved.


## Data quality checks

Before proceeding with the analysis, the collected dataset was validated to ensure completeness and consistency.

The validation confirmed that:

- all observations were successfully retrieved from the Wikimedia Pageviews API;
- no missing values were present;
- each selected article contained the same number of daily observations (551 days);
- the dataset covers the entire study period from May 14, 2023 to November 14, 2024.

1. Structure

In [33]:
raw_pageviews.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8265 entries, 0 to 8264
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype         
---  ------   --------------  -----         
 0   date     8265 non-null   datetime64[ns]
 1   article  8265 non-null   object        
 2   views    8265 non-null   int64         
dtypes: datetime64[ns](1), int64(1), object(1)
memory usage: 193.8+ KB


2. Summary statistics

In [36]:
raw_pageviews["views"].describe()

count    8265.000000
mean     1084.584634
std       539.695652
min       142.000000
25%       586.000000
50%      1086.000000
75%      1471.000000
max      7000.000000
Name: views, dtype: float64

3. Number of observations

In [37]:
validation = (
    raw_pageviews
    .groupby("article")
    .size()
    .reset_index(name="days")
)

validation

,article,days
0,Anemia,551
1,Anxiety,551
2,Asthma,551
3,Common cold,551
4,Depression (mood),551
5,Diabetes,551
6,Fatigue,551
7,Headache,551
8,Hypertension,551
9,Influenza,551


## Research hypothesis

We define May 14, 2024 as the intervention date because it corresponds to Google's broader rollout of AI Overviews in Search.

The research hypothesis is:

> The introduction of Google AI Overviews reduced pageviews to Wikipedia articles that are frequently used to answer informational search queries.